# 🎓 Student Project Recommendation ML System
## End-to-End Pipeline, Model Training & Accuracy Evaluation

This Jupyter Notebook consolidates the **entire Machine Learning pipeline** into a single, self-contained interactive notebook:

### 📋 Notebook Sections:
1. **Dependencies & Setup**: Import essential scientific and ML packages.
2. **Data Ingestion & Inspection**: Inspect dataset shape, data types, missing values, and target distributions.
3. **Data Cleaning**: Remove duplicates and validate required features.
4. **Data Preprocessing Pipeline**: Custom TF-IDF vectorizer for `Current_Skills`, Ordinal Encoding for categorical features, and numeric imputation.
5. **Train-Test Split**: Leakage-free 80/20 train/test split.
6. **Model Training & Comparison**: Compare **Logistic Regression** vs. **Random Forest Classifier**.
7. **Comprehensive Model Accuracy & Metrics**:
   - Training Accuracy & Testing Accuracy
   - Weighted Precision, Recall, and F1-Score
   - Generalization Gap & Overfitting Diagnostics
   - Detailed Classification Report for all 50+ project categories
8. **Visualizations**: Accuracy comparison bar plot & Confusion Matrix heatmap with Matplotlib.
9. **Regression Experiment**: Predicting `Available_Hours_Per_Week` via Random Forest Regressor.
10. **Interactive Prediction System**: Live recommendation function for custom student profiles.


In [ ]:
# Optional: uncomment to install dependencies if not already installed
# %pip install pandas numpy scikit-learn matplotlib joblib

import os
import sys
import json
import warnings
warnings.filterwarnings('ignore') # Clean display

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print(f"Pandas version      : {pd.__version__}")
print(f"NumPy version       : {np.__version__}")
import sklearn
print(f"Scikit-learn version: {sklearn.__version__}")
print("✓ All libraries loaded successfully.")


---
## 1. Load and Inspect Dataset
We load `student_recommendation_10000_no_difficulty.csv`. The code includes dynamic path detection so it works seamlessly from any directory.


In [ ]:
# Auto-detect CSV location
possible_paths = [
    "data/student_recommendation_10000_no_difficulty.csv",
    "../data/student_recommendation_10000_no_difficulty.csv",
    "student_ml_from_scratch/data/student_recommendation_10000_no_difficulty.csv",
    os.path.join(os.getcwd(), "data", "student_recommendation_10000_no_difficulty.csv"),
    os.path.join(os.getcwd(), "..", "data", "student_recommendation_10000_no_difficulty.csv"),
    os.path.join(os.getcwd(), "student_ml_from_scratch", "data", "student_recommendation_10000_no_difficulty.csv"),
]

raw_csv_path = None
for p in possible_paths:
    if os.path.exists(p):
        raw_csv_path = p
        break

if raw_csv_path is None:
    raise FileNotFoundError("Could not find student_recommendation_10000_no_difficulty.csv in expected directories.")

print(f"Dataset path: {raw_csv_path}")
df = pd.read_csv(raw_csv_path)

print(f"\n--- Dataset Dimensions ---")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

print(f"\n--- First 5 Records ---")
display(df.head()) if 'display' in globals() else print(df.head())

print(f"\n--- Column Summary & Missing Values ---")
summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Missing (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
print(summary)


---
## 2. Data Cleaning & Target Distribution
Remove duplicate records and verify feature integrity and class distribution.


In [ ]:
# Deduplication
initial_rows = len(df)
n_dups = df.duplicated().sum()
df_clean = df.drop_duplicates().copy()
print(f"Removed {n_dups} duplicate rows ({initial_rows:,} -> {len(df_clean):,})")

# Feature definitions
CATEGORICAL_FEATURES = ["Year", "Department", "Interest"]
TEXT_FEATURE = "Current_Skills"
NUMERIC_FEATURE = "Available_Hours_Per_Week"
TARGET_COLUMN = "Recommended_Project"
ALL_FEATURES = CATEGORICAL_FEATURES + [TEXT_FEATURE, NUMERIC_FEATURE]

# Drop rows missing critical columns
df_clean = df_clean.dropna(subset=ALL_FEATURES + [TARGET_COLUMN])
print(f"Cleaned dataset rows: {len(df_clean):,}")

# Inspect Target Column
target_counts = df_clean[TARGET_COLUMN].value_counts()
n_classes = df_clean[TARGET_COLUMN].nunique()
print(f"\n--- Target Analysis ---")
print(f"Target column        : {TARGET_COLUMN}")
print(f"Total unique classes : {n_classes}")
print(f"Most frequent class  : '{target_counts.idxmax()}' ({target_counts.max()} samples)")
print(f"Least frequent class : '{target_counts.idxmin()}' ({target_counts.min()} samples)")

print(f"\nTop 10 recommended projects:")
for idx, (proj, count) in enumerate(target_counts.head(10).items(), 1):
    print(f"  {idx:2d}. {proj:<45} : {count} samples")


---
## 3. Preprocessing Pipeline Definition
We construct a Scikit-Learn `ColumnTransformer`:
- **Categorical** (`Year`, `Department`, `Interest`): `SimpleImputer` + `OrdinalEncoder`
- **Numeric** (`Available_Hours_Per_Week`): `SimpleImputer(median)`
- **Free-Text** (`Current_Skills`): Flattens 2D arrays to 1D and vectorizes via `TfidfVectorizer(max_features=200, ngram_range=(1,2))`


In [ ]:
class FlattenThenTfidf(BaseEstimator, TransformerMixin):
    """
    ColumnTransformer passes a 2-D array/DataFrame even for a single column.
    This custom transformer flattens it to 1-D strings for TfidfVectorizer.
    """
    def __init__(self, max_features=200, ngram_range=(1, 2), sublinear_tf=True):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.sublinear_tf = sublinear_tf
        self._tfidf = TfidfVectorizer(
            max_features=self.max_features,
            ngram_range=self.ngram_range,
            sublinear_tf=self.sublinear_tf,
        )

    def _flatten(self, X):
        if hasattr(X, "iloc"):
            col = X.iloc[:, 0]
        elif hasattr(X, "ndim") and X.ndim > 1:
            col = X[:, 0]
        else:
            col = X
        return [str(v) if pd.notna(v) else "" for v in col]

    def fit(self, X, y=None):
        self._tfidf.fit(self._flatten(X))
        return self

    def transform(self, X):
        return self._tfidf.transform(self._flatten(X)).toarray()


def build_preprocessor() -> ColumnTransformer:
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
    ])

    text_pipe = FlattenThenTfidf(
        max_features=200,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    return ColumnTransformer(
        transformers=[
            ("cat",  cat_pipe,  CATEGORICAL_FEATURES),
            ("num",  num_pipe,  [NUMERIC_FEATURE]),
            ("text", text_pipe, [TEXT_FEATURE]),
        ],
        remainder="drop",
        sparse_threshold=0,
    )

print("✓ Preprocessing pipeline definition constructed.")


---
## 4. Train-Test Split (80% Train, 20% Test)
Splits data into training (8,000 samples) and testing (2,000 samples). Stratification is applied when class minimum counts allow.


In [ ]:
X = df_clean[ALL_FEATURES].copy()
y = df_clean[TARGET_COLUMN].copy()

min_count = y.value_counts().min()
stratify_val = y if min_count >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=stratify_val
)

print(f"Total dataset samples : {len(X):,}")
print(f"Training split samples: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing split samples : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"Stratification active : {stratify_val is not None}")


---
## 5. Model Training & Comparison: Logistic Regression vs. Random Forest
We train both algorithms end-to-end and compare their performance.


In [ ]:
print("=" * 65)
print("             TRAINING MACHINE LEARNING MODELS")
print("=" * 65)

# 1. Logistic Regression
print("\n[1/2] Training Logistic Regression Baseline...")
lr_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs", C=1.0))
])
lr_pipeline.fit(X_train, y_train)

lr_train_acc = accuracy_score(y_train, lr_pipeline.predict(X_train))
lr_test_acc  = accuracy_score(y_test,  lr_pipeline.predict(X_test))
print(f"      -> Train Accuracy: {lr_train_acc*100:.2f}% | Test Accuracy: {lr_test_acc*100:.2f}%")

# 2. Random Forest Classifier
print("\n[2/2] Training Random Forest Classifier...")
rf_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", RandomForestClassifier(n_estimators=150, max_depth=None, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf_pipeline.predict(X_train))
rf_test_acc  = accuracy_score(y_test,  rf_pipeline.predict(X_test))
print(f"      -> Train Accuracy: {rf_train_acc*100:.2f}% | Test Accuracy: {rf_test_acc*100:.2f}%")

# Comparison Table
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest Classifier"],
    "Train Accuracy (%)": [f"{lr_train_acc*100:.2f}%", f"{rf_train_acc*100:.2f}%"],
    "Test Accuracy (%)": [f"{lr_test_acc*100:.2f}%", f"{rf_test_acc*100:.2f}%"],
    "Train-Test Gap (%)": [f"{(lr_train_acc - lr_test_acc)*100:.2f}%", f"{(rf_train_acc - rf_test_acc)*100:.2f}%"],
})
print("\n" + "=" * 65)
print("               MODEL ACCURACY COMPARISON")
print("=" * 65)
print(comparison.to_string(index=False))

# Select Best Model
if rf_test_acc >= lr_test_acc:
    best_pipeline = rf_pipeline
    best_name = "Random Forest"
    best_acc = rf_test_acc
else:
    best_pipeline = lr_pipeline
    best_name = "Logistic Regression"
    best_acc = lr_test_acc

print(f"\n🏆 Winning Model: {best_name} with {best_acc*100:.2f}% Test Accuracy!")


---
## 6. Model Accuracy & Comprehensive Evaluation Metrics
Here we evaluate the winning model in full detail:
- **Training Accuracy**
- **Testing Accuracy**
- **Weighted Precision**
- **Weighted Recall**
- **Weighted F1-Score**
- **Overfitting Diagnostics**
- **Classification Report**


In [ ]:
y_pred_train = best_pipeline.predict(X_train)
y_pred_test  = best_pipeline.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test,  y_pred_test)
precision = precision_score(y_test, y_pred_test, average="weighted", zero_division=0)
recall    = recall_score(y_test,    y_pred_test, average="weighted", zero_division=0)
f1        = f1_score(y_test,        y_pred_test, average="weighted", zero_division=0)
gap       = train_acc - test_acc
cm        = confusion_matrix(y_test, y_pred_test)

print("=" * 68)
print(f"   FULL ACCURACY & EVALUATION REPORT ({best_name})")
print("=" * 68)
print(f"  • Training Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"  • Testing Accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  • Weighted Precision: {precision:.4f}  ({precision*100:.2f}%)")
print(f"  • Weighted Recall   : {recall:.4f}  ({recall*100:.2f}%)")
print(f"  • Weighted F1-Score : {f1:.4f}  ({f1*100:.2f}%)")
print(f"  • Train-Test Gap    : {gap:.4f}  ({gap*100:.2f}%)")

print("\n--- Generalization & Overfitting Check ---")
if gap > 0.15:
    print(f"  [!] Gap = {gap:.4f} - Potential Overfitting detected.")
elif abs(gap) <= 0.05:
    print(f"  [OK] Gap = {gap:.4f} - Excellent generalization! No overfitting.")
else:
    print(f"  [i] Gap = {gap:.4f} - Mild gap; within normal limits.")

print("\n--- Classification Report (First 25 Lines) ---")
clf_report = classification_report(y_test, y_pred_test, zero_division=0)
print("\n".join(clf_report.splitlines()[:25]))
print("... [Complete report covers all 50+ classes]")


---
## 7. Performance Visualizations
Visualizing accuracy comparison across models and the confusion matrix heatmap.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Model Accuracy Bar Chart
models = ['Logistic Regression', 'Random Forest']
train_vals = [lr_train_acc * 100, rf_train_acc * 100]
test_vals  = [lr_test_acc * 100,  rf_test_acc * 100]

x = np.arange(len(models))
width = 0.35

rects1 = axes[0].bar(x - width/2, train_vals, width, label='Train Accuracy', color='#2563eb', alpha=0.9)
rects2 = axes[0].bar(x + width/2, test_vals,  width, label='Test Accuracy',  color='#059669', alpha=0.9)

axes[0].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Training vs Testing Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, fontsize=11, fontweight='bold')
axes[0].set_ylim(85, 105)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].legend(loc='lower right', fontsize=11)

for bar in rects1:
    h = bar.get_height()
    axes[0].annotate(f'{h:.2f}%', xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in rects2:
    h = bar.get_height()
    axes[0].annotate(f'{h:.2f}%', xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold')

# Subplot 2: Confusion Matrix Heatmap
im = axes[1].imshow(cm, cmap='Blues', interpolation='nearest')
axes[1].set_title(f'Confusion Matrix Heatmap ({cm.shape[0]} Classes)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Class Index', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Class Index', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


---
## 8. Regression Experiment: Predict Available Hours Per Week
Evaluating whether student characteristics (`Year`, `Department`, `Interest`, `Skills`, `Project`) can predict `Available_Hours_Per_Week`.


In [ ]:
reg_target = "Available_Hours_Per_Week"
reg_features = ["Year", "Department", "Current_Skills", "Interest", "Recommended_Project"]

available_reg_cols = [c for c in reg_features if c in df_clean.columns]
X_reg = df_clean[available_reg_cols].copy()
y_reg = df_clean[reg_target].copy()

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)

reg_cat_cols = [c for c in available_reg_cols if c != "Current_Skills"]
reg_transformers = [
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), reg_cat_cols),
    ("text", FlattenThenTfidf(max_features=100), ["Current_Skills"])
]

reg_preprocessor = ColumnTransformer(transformers=reg_transformers, remainder="drop")
reg_pipeline = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training Regression Model...")
reg_pipeline.fit(X_tr, y_tr)
y_pred_reg = reg_pipeline.predict(X_te)

mae = mean_absolute_error(y_te, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_te, y_pred_reg))
r2 = r2_score(y_te, y_pred_reg)

print("=" * 65)
print("             REGRESSION EXPERIMENT METRICS")
print("=" * 65)
print(f"  • Target Feature              : {reg_target}")
print(f"  • Mean Absolute Error (MAE)   : {mae:.3f} hours")
print(f"  • Root Mean Squared Error     : {rmse:.3f} hours")
print(f"  • R² Score                    : {r2:.4f}")
if r2 < 0.15:
    print("  • Insight                     : R² is close to 0, indicating available hours were generated independently.")
else:
    print(f"  • Insight                     : Meaningful correlation observed (R² = {r2:.4f}).")


---
## 9. Interactive Prediction System
Test project recommendations on single student profiles in real-time.


In [ ]:
def recommend_project(year, department, current_skills, interest, available_hours):
    """Predict the recommended project for a student profile."""
    student_df = pd.DataFrame([{
        "Year": str(year),
        "Department": str(department),
        "Current_Skills": str(current_skills),
        "Interest": str(interest),
        "Available_Hours_Per_Week": float(available_hours)
    }], columns=ALL_FEATURES)
    
    return best_pipeline.predict(student_df)[0]

sample_students = [
    {"year": "2nd Year", "department": "IT",  "current_skills": "Python Basics, pandas, data analysis", "interest": "Data Science", "available_hours": 15},
    {"year": "1st Year", "department": "CSE", "current_skills": "HTML, CSS, JavaScript, React",         "interest": "Web Development", "available_hours": 10},
    {"year": "3rd Year", "department": "ECE", "current_skills": "C Basics, Arduino, IoT sensors",       "interest": "Electronics", "available_hours": 20},
    {"year": "4th Year", "department": "CSE", "current_skills": "Deep Learning, PyTorch, Computer Vision", "interest": "Artificial Intelligence", "available_hours": 25},
]

print("=" * 75)
print("             DEMO: REAL-TIME STUDENT RECOMMENDATIONS")
print("=" * 75)
for i, s in enumerate(sample_students, 1):
    rec = recommend_project(s["year"], s["department"], s["current_skills"], s["interest"], s["available_hours"])
    print(f"\nStudent #{i}:")
    print(f"  Profile : {s['year']} | {s['department']} | Interest: {s['interest']} ({s['available_hours']} hrs/week)")
    print(f"  Skills  : {s['current_skills']}")
    print(f"  🎯 Recommended Project: {rec}")


### ✏️ Test Your Own Student Profile:
Change the values below and run this cell to see your personalized project recommendation!


In [ ]:
# Edit these variables with any profile you want to test:
my_year = "3rd Year"
my_department = "CSE"
my_skills = "Python, Machine Learning, Scikit-learn, SQL"
my_interest = "Artificial Intelligence"
my_hours = 18

project = recommend_project(
    year=my_year,
    department=my_department,
    current_skills=my_skills,
    interest=my_interest,
    available_hours=my_hours
)

print("=" * 60)
print(f"Student Profile: {my_year} {my_department}")
print(f"Skills         : {my_skills}")
print(f"Interest       : {my_interest} ({my_hours} hrs/week)")
print("-" * 60)
print(f"⭐ Recommended Project: {project}")
print("=" * 60)
